In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "id": "99c1d50a-5bbc-473b-aa6b-aa26501038d0",
   "metadata": {},
   "source": [
    "using:",
    "",
    "https://github.com/henrysky/astroNN",
    "",
    "data: https://astronn.readthedocs.io/en/latest/galaxy10.html",
    ""
   ]
  },
  {
   "cell_type": "markdown",
   "id": "4fdede67-1f86-4507-a04e-4d91f6ad34dc",
   "metadata": {},
   "source": [
    "## cleaning data",
    "### normalising pixel values:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "9e2709c8-3045-4f8b-a59c-2c7e97e9ed5f",
   "metadata": {},
   "outputs": [],
   "source": [
    "import h5py",
    "",
    "file_path=\"Galaxy10.h5\"    "
   ]
  },
  {
   "cell_type": "markdown",
   "id": "d69c246d-0146-4ae1-b771-12f7577e1b50",
   "metadata": {},
   "source": [
    "### Let us study the data:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "1259c65d-b91e-490e-b8a0-0066ec12ea12",
   "metadata": {},
   "outputs": [],
   "source": [
    "with h5py.File(file_path,'r') as f:",
    "    print(\"Keys in the file:\", list(f.keys()))",
    "",
    "    images=f['images'][:]",
    "    print(f\"Images shape: {images.shape}\")",
    "",
    "    labels=f['ans'][:]",
    "    print(f\"Labels shape:{labels.shape}\")",
    "    print(f\"Unique labels: {set(labels)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "af171f55-0a25-4657-abd5-45d17766e557",
   "metadata": {},
   "source": [
    "#### So, there are:",
    "1. 21785 images with 21785 labels",
    "2. each image is 69x69 pixels",
    "3. images have 3 channels (likely RGB channels)",
    "4. and  there 10 classes of labels (0-9) which means there are 10 types of galaxies in the data."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2249618e-a667-4a04-9108-a9ce19182910",
   "metadata": {},
   "source": [
    "### Normalising Picture Values",
    "",
    "Normalised Picture Value = Pixel Value / 255.0"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a78a2fc4-093e-4250-afe4-e265875a9295",
   "metadata": {},
   "outputs": [],
   "source": [
    "# normalised_images = images/255.0",
    "",
    "# print(f\"Normalized image shape: {normalised_images.shape}\")",
    "# print(f\"Pixel value range: {normalised_images.min()} to {normalised_images.max()}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "2b5870c7-9e68-4342-8e54-de59b2247635",
   "metadata": {},
   "source": [
    "We can optionally save the normalised image data:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "fac918e8-c4bc-437e-b675-2f0e9fb491c5",
   "metadata": {},
   "outputs": [],
   "source": [
    "# normalised_image_file=\"Galaxy10_normalized.h5\"",
    "",
    "# with h5py.File(normalised_image_file, 'w') as f:",
    "#     f.create_dataset('images',data=normalised_images)",
    "#     print(f\"Saved to {normalised_image_file}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "04494575-e997-4450-963e-1375b52a9e5a",
   "metadata": {},
   "source": [
    "### Now, we split the data into training and validation sets"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "84291b99-9e93-4687-ae11-3cf36470ea6b",
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.model_selection import train_test_split",
    "",
    "with h5py.File(file_path, 'r') as f:",
    "    labels=f['ans'][:]",
    "",
    "X_train, X_val, y_train, y_val = train_test_split(images, labels, test_size=0.2, random_state=42)",
    "",
    "print(f\"Training set shape: {X_train.shape}\Validation set shape:{X_val.shape}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5c0528e8-86dd-4673-b7ce-acd8423ca3dc",
   "metadata": {},
   "source": [
    "### Building a simple CNN"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "700ddeeb-7574-4881-b261-f533fbcf7d0d",
   "metadata": {},
   "outputs": [],
   "source": [
    "import tensorflow as tf"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "39949432-d3e7-495f-add0-257f78510f8c",
   "metadata": {},
   "outputs": [],
   "source": [
    "from tensorflow.keras.models import Sequential",
    "from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Input"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "14b13440-fe6f-47d3-8d67-18fb774c32d8",
   "metadata": {},
   "source": [
    "Defining the CNN model:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "5fcbdc70-52e4-4dbc-95c6-ee3b759a3c0f",
   "metadata": {},
   "outputs": [],
   "source": [
    "model = Sequential([",
    "    Input(shape=(64, 64, 3)),  ",
    "    Conv2D(32, (3, 3), activation='relu'),",
    "    MaxPooling2D((2, 2)), #halves the pooling window",
    "    Conv2D(64, (3, 3), activation='relu'),",
    "    MaxPooling2D((2, 2)),",
    "    Flatten(),",
    "    Dense(128, activation='relu'),",
    "    Dense(10, activation='softmax')  ",
    "])"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "f553dea0-c624-4c3c-a2dc-2aaaee34939f",
   "metadata": {},
   "source": [
    "How the Model Works",
    "",
    "    1. The input image (64, 64, 3) passes through the convolutional layers to extract features like edges and patterns",
    "    2. MaxPooling layers reduce the size of the feature maps, keeping only the most important information.",
    "    3. The flatten layer prepares the data for the fully connected dense layers.",
    "    4. The dense layers process the features and output probabilities for each of the 3 classes.",
    "",
    "Needed to add another dense layer due to error."
   ]
  },
  {
   "cell_type": "markdown",
   "id": "138cf96b-6a58-4e3a-ab17-3356d5075411",
   "metadata": {},
   "source": [
    "This model is designed for a classification task with 3 classes.",
    "It works well for small image datasets and can be trained quickly."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d8efeabc-dcfa-49e7-bafa-22a992c9623f",
   "metadata": {},
   "outputs": [],
   "source": [
    "model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])",
    "model.summary()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "e022610f-9d10-4a38-844d-566e7b6c1f54",
   "metadata": {},
   "source": [
    "### Training the model:"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "a2662da0-4025-409d-b649-3776f48b69b7",
   "metadata": {},
   "outputs": [],
   "source": [
    "from tensorflow.keras.preprocessing.image import ImageDataGenerator",
    "from tensorflow.image import resize",
    "import numpy as np"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "3a484156-af7c-4e68-a874-1a52b917fdbd",
   "metadata": {},
   "outputs": [],
   "source": [
    "X_train = np.array([resize(img, (64, 64)).numpy() for img in X_train])",
    "X_val = np.array([resize(img, (64, 64)).numpy() for img in X_val]) #added to make even division"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "66627ce0-ad15-46e8-9fe1-cdc28e050247",
   "metadata": {},
   "outputs": [],
   "source": [
    "datagen = ImageDataGenerator(",
    "    rotation_range=20,",
    "    width_shift_range=0.2,",
    "    height_shift_range=0.2,",
    "    horizontal_flip=True",
    ")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "02c8e673-7cd7-48e4-94e7-9bc1af39d5a5",
   "metadata": {},
   "outputs": [],
   "source": [
    "train_gen=datagen.flow(X_train,y_train,batch_size=32)",
    "val_gen=datagen.flow(X_val,y_val,batch_size=32)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "d84aaa4b-1479-4e0b-9700-73b052c9aac5",
   "metadata": {},
   "outputs": [],
   "source": [
    "history=model.fit(train_gen,validation_data=val_gen,epochs=10)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "918bf47f-f4c2-479f-8dc9-8e32d384ff6e",
   "metadata": {},
   "outputs": [],
   "source": [
    "import matplotlib.pyplot as plt"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "id": "c0b9142d-7a2d-4bc8-964e-91cad5cfe61e",
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.plot(history.history['accuracy'], label='Training Accuracy')",
    "plt.plot(history.history['val_accuracy'], label='Validation Accuracy')",
    "plt.xlabel('Epochs') ",
    "plt.ylabel('Accuracy')",
    "plt.title(\"Training and Validation Accuracy of Custom CNN\")",
    "plt.suptitle(\"The plot shows that the model gets better at identifying galaxies over time (training accuracy) and then does a good job unseen data (validation accuracy)\")",
    "plt.legend()",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "id": "5a8b49aa-5ba9-42fc-80da-5376b6d32040",
   "metadata": {},
   "source": [
    "### In the beginning we separated the data into training vs validation. The bar chart above shows how well the model performed after training on labelled data and then went on to classify validation data itself. Since we know the labels for validation data, we can compare the model's classification vs default classification and get a validation accuracy result. "
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3 (ipykernel)",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.12.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

NameError: name 'null' is not defined